In [1]:
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
import numpy as np
import pandas as pd 
from scipy.spatial.transform import Rotation as R
from scipy.spatial.distance import pdist, squareform
from pathlib import Path 
import tqdm
import umap



def xyz_to_rdkit_mol(xyz_file, total_charge):
    """
    Convert an XYZ file with explicit hydrogens to an RDKit Mol,
    preserving 3D coordinates and determining connectivity automatically.
    

    Parameters:
        xyz_file: path to XYZ file
        total_charge: total molecular charge (default 0)
    
    Returns:
        RDKit Mol object
    """
    # --- Read XYZ ---
    with open(xyz_file) as f:
        lines = f.readlines()[2:]  # skip atom count + comment
    atoms, coords = [], []

    for line in lines:
        parts = line.split()
        atoms.append(parts[0])
        coords.append([float(x) for x in parts[1:4]])
    coords = np.array(coords)
    
    # --- Create empty molecule with atoms ---
    mol = Chem.RWMol()
    z = [Chem.GetPeriodicTable().GetAtomicNumber(a) for a in atoms]
    for Zi in z:
        mol.AddAtom(Chem.Atom(Zi))
    
    # --- Add coordinates ---
    conf = Chem.Conformer(len(coords))
    for i, pos in enumerate(coords):
        conf.SetAtomPosition(i, pos)
    mol.AddConformer(conf)
    
    # --- Determine connectivity using RDKit's bond perception ---
    Chem.rdDetermineBonds.DetermineConnectivity(mol, charge=total_charge)
    
    # --- Sanitize molecule ---
    Chem.SanitizeMol(mol)
    
    return mol



def align_and_rmsd_numpy(X, Y, return_aligned=False):
    X_center = X.mean(axis=0)
    Y_center = Y.mean(axis=0)
    Xc = X - X_center
    Yc = Y - Y_center

    rot, _ = R.align_vectors(Xc, Yc)
    Y_aligned = rot.apply(Yc) + X_center
    rmsd = np.sqrt(np.mean(np.sum((X - Y_aligned)**2, axis=1)))

    if return_aligned:
        return rmsd, Y_aligned
    else:
        return rmsd


def update_mol_coordinates_copy(mol, new_coords):
    """
    Return a copy of an RDKit molecule with updated 3D coordinates.
    
    Parameters:
        mol: RDKit Mol object (must have same number of atoms as new_coords)
        new_coords: (N,3) numpy array of new coordinates
    
    Returns:
        new_mol: RDKit Mol object copy with updated coordinates
    """
    if new_coords.shape[0] != mol.GetNumAtoms():
        raise ValueError("Number of coordinates must match number of atoms")
    
    # --- Make a true copy ---
    new_mol = Chem.Mol(mol)  # copy molecule
    # Remove old conformers
    for conf_id in [c.GetId() for c in new_mol.GetConformers()]:
        new_mol.RemoveConformer(conf_id)
    
    # --- Add new conformer ---
    conf = Chem.Conformer(new_mol.GetNumAtoms())
    for i, pos in enumerate(new_coords):
        conf.SetAtomPosition(i, pos)
    new_mol.AddConformer(conf)
    
    return new_mol

def reflect_axis(geometry, reflection): 
    reflected_geom = geometry * reflection 
    return reflected_geom



def numpy_geom(mol):
    conf = mol.GetConformer()
    return np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])


In [5]:
raw_spawns_folder = Path('/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene')
meci_for_alignment = Path('/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/meci/ethylene/0000_2.xyz')

In [6]:

use_reflections = True 



geometries = {}
families = {}
templates = {}
stems = []

REFLECTIONS = np.array(
    [
        [1, 1, 1],
        [-1, 1, 1],
        [1, -1, 1],
        [1, 1, -1],
        [-1, 1, -1],
        [-1, -1, 1],
        [1, -1, -1],
        [-1, -1, -1],
    ]
)


meci_mol=xyz_to_rdkit_mol(meci_for_alignment, total_charge=0)
meci_smiles = Chem.MolToSmiles(Chem.Mol(meci_mol), canonical=True)
templates[meci_smiles]= meci_mol  
geometries[meci_smiles] = {}

#uses MECI for initial alignment, will use first instance of each new smiles for subsequent alignment. 

for x in tqdm.tqdm(list(raw_spawns_folder.glob('*'))):
   

    mol = xyz_to_rdkit_mol(x, total_charge=0)
    smiles = Chem.MolToSmiles(Chem.Mol(mol), canonical=True)
    #print(smiles)
    stems.append(x.stem)

    if smiles not in templates:
        templates[smiles] = mol
        geometries[smiles] = {}
        geometries[smiles][x] = mol
        print(x)
    else:
        template = templates[smiles]
        template_geom = numpy_geom(template)
        mol_geom = numpy_geom(mol)

        matches = mol.GetSubstructMatches(template, uniquify=False)

        if not matches:
            continue

        rmsds = []
        for match in matches:
            swapped = mol_geom[list(match)]  


            
            if use_reflections == True: 
                for reflection in REFLECTIONS: 
                    swapped_reflection = reflect_axis(swapped, reflection)
                    rmsd, aligned_coords = align_and_rmsd_numpy(template_geom, swapped_reflection, return_aligned=True)
                    rmsds.append((rmsd, aligned_coords))


            else:


                rmsd, aligned_coords = align_and_rmsd_numpy(template_geom, swapped, return_aligned=True)
                rmsds.append((rmsd, aligned_coords))




        best_rmsd, best_aligned = min(rmsds, key=lambda t: t[0])

        geometries[smiles][x] = update_mol_coordinates_copy(mol, best_aligned)

  1%|          | 14/2557 [00:00<00:18, 135.38it/s]

/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0648_2.xyz
/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0299_2.xyz
/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0572_4.xyz


  2%|▏         | 43/2557 [00:00<00:18, 135.18it/s]

/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0535_6.xyz


 41%|████      | 1038/2557 [00:07<00:10, 140.47it/s]

/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0341_3.xyz
/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0649_6.xyz


 78%|███████▊  | 1989/2557 [00:14<00:04, 134.82it/s]

/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/spawn/ethylene/0190_2.xyz


100%|██████████| 2557/2557 [00:18<00:00, 136.98it/s]


In [7]:
for key, x  in geometries.items():
    print(f"{key}   {len(x.keys())}")

[H][C]([H])[C]([H])[H]   822
[H].[H][C][C]([H])[H]   853
[H][C][H].[H][C][H]   125
[H][C]C([H])([H])[H]   729
[H][C][C][H].[H][H]   9
[H][C].[H][C]([H])[H]   11
[H].[H]C([H])([H])[C]   6
[H].[H].[H][C][C][H]   2


In [8]:
output_main_dir = Path('/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/aligned_geometries/substruct_match/ethylene_smiles')
output_main_dir.mkdir(exist_ok=True, parents=True)
counter = 0
for smiles, mol_dict in geometries.items():
    smiles_dir = output_main_dir / str(counter)
    smiles_dir.mkdir(exist_ok=True, parents=True)
    counter = counter + 1
    for idx, mol in mol_dict.items():
        output_geom = smiles_dir / f"{idx.name}"
        Chem.MolToXYZFile(mol, str(output_geom))

In [ ]:
#### import meci classification, compare adjusted rand membership to labels generated above 